# Feature Validation & Cut Point Analysis

Runs the full validation pipeline using `feature_extractor.analysis`.

In [ ]:
import logging
from pathlib import Path

from feature_extractor.config.schemas import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

config = load_config()

CACHE_DIR = config.cache.cache_path
RESULTS_DIR = CACHE_DIR / "results"
FEATURE_MATRIX = CACHE_DIR / "feature_matrix.parquet"

In [ ]:
from feature_extractor.analysis.runner import run_validation_pipeline

results = run_validation_pipeline(
    feature_matrix_path=FEATURE_MATRIX,
    output_dir=RESULTS_DIR,
    target="score",
    corr_threshold=0.85,
    discrete_scores=not config.scoring.normalize,
)

In [ ]:
import pandas as pd

print(f"Pruned features: {len(results['pruned_features'])}")
print(f"Cut points derived: {len(results['feature_cuts'])}")
print()

ranking = results["cross_ranking"]
consistent = ranking[ranking["sign_consistent"] & (ranking["n_significant"] >= ranking["n_prompts"] - 1)]
print(f"Cross-prompt consistent features ({len(consistent)}/{len(ranking)}):")
consistent[["feature", "module", "min_abs_rho", "mean_abs_rho", "n_significant", "sign_consistent", "direction"]]

In [ ]:
cuts = results["feature_cuts"]
cut_cols = [c for c in cuts.columns if c.startswith("cut_")]
print(f"Feature cut points ({len(cuts)} features, {len(cut_cols)} cuts):")
display_cols = ["module", "feature", "min_abs_rho"] + cut_cols + ["direction"]
cuts[display_cols].sort_values(["module", "feature"])

In [ ]:
if results["dropped_pairs"]:
    print(f"Correlation pruning dropped {len(results['dropped_pairs'])} features:")
    pd.DataFrame(results["dropped_pairs"])
else:
    print("No features dropped by correlation pruning.")